# ResT signature scanner & verifier (Colab)

Runtime → Change runtime type → **T4 GPU**, then run all cells. The last cell prints a
`https://….gradio.live` link: open it on your phone (the camera needs HTTPS). The link
lasts until this session stops (max 72 h).

**Mock-test use only**: ~75% accuracy on GPDS, and phone photos are out of domain.

In [ ]:
!nvidia-smi

## 1. Clone the repo (code only; LFS files are skipped)

In [ ]:
%cd /content
!rm -rf /content/ResT
!GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/buixuanhieu1999/ResT.git /content/ResT
!mkdir -p /content/ResT/webapp
%cd /content/ResT

## 2. Write the web app files
Generated from `webapp/` by `make_notebook.py`, so this works even before `webapp/` is pushed.

In [ ]:
%%writefile /content/ResT/webapp/cleanup.py
"""Shadow-robust signature extraction for phone photos.

The model's own preprocessing (``train_cedar.SignatureTransform``) applies one
global Otsu threshold, which is fine for flat GPDS scans but turns shadows and
lighting gradients in phone photos into "ink". This module flattens the
illumination first, so only real strokes survive, and returns a clean
black-on-white image that is then handed to SignatureTransform unchanged.
"""

from __future__ import annotations

from typing import NamedTuple

import cv2
import numpy as np

MAX_LONG_SIDE = 1600
INK_COLOURS = {
    "black": (0, 0, 0),
    "blue": (16, 42, 150),
}


class CleanResult(NamedTuple):
    rgb: np.ndarray  # HxWx3 uint8, ink on white (this is what the model sees)
    rgba: np.ndarray  # HxWx4 uint8, ink on transparent (for download)
    found_ink: bool


def _odd(value: float, minimum: int = 3) -> int:
    size = max(minimum, int(round(value)))
    return size if size % 2 else size + 1


def _to_rgb(image: np.ndarray) -> np.ndarray:
    """Accept gray, RGB or RGBA (any dtype Gradio might send) and return RGB uint8."""

    image = np.asarray(image)
    if image.dtype != np.uint8:
        scale = 255.0 if image.dtype.kind == "f" and image.max() <= 1.0 else 1.0
        image = np.clip(image * scale, 0, 255).astype(np.uint8)
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    if image.shape[2] == 4:
        # Composite transparent pixels onto white so they don't read as ink.
        alpha = image[..., 3:4].astype(np.float32) / 255.0
        rgb = image[..., :3].astype(np.float32) * alpha + 255.0 * (1.0 - alpha)
        return rgb.astype(np.uint8)
    return image[..., :3]


def _downscale(rgb: np.ndarray) -> np.ndarray:
    height, width = rgb.shape[:2]
    long_side = max(height, width)
    if long_side <= MAX_LONG_SIDE:
        return rgb
    scale = MAX_LONG_SIDE / long_side
    size = (max(1, round(width * scale)), max(1, round(height * scale)))
    return cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)


def darkness_map(gray: np.ndarray) -> np.ndarray:
    """Per-pixel ink darkness relative to the locally estimated paper brightness."""

    long_side = max(gray.shape)
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (_odd(long_side / 90),) * 2
    )
    # Dilation erases thin dark strokes; the median smooths what is left into a
    # paper/shadow illumination estimate.
    background = cv2.dilate(gray, kernel)
    background = cv2.medianBlur(background, min(_odd(long_side / 25), 255))
    background = np.maximum(background.astype(np.float32), 1.0)
    return np.clip(1.0 - gray.astype(np.float32) / background, 0.0, 1.0)


def _ink_threshold(darkness: np.ndarray, sensitivity: float) -> float:
    candidates = darkness[darkness > 0.04]
    if candidates.size < 50:
        threshold = 0.25
    else:
        values = np.round(candidates * 255.0).astype(np.uint8).reshape(-1, 1)
        otsu, _ = cv2.threshold(values, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        threshold = float(otsu) / 255.0
    return max(threshold, 0.10) / max(sensitivity, 1e-3)


def _keep_strokes(
    mask: np.ndarray, min_speck: float, remove_border_blobs: bool
) -> np.ndarray:
    height, width = mask.shape
    long_side = max(height, width)
    min_area = min_speck * (long_side / 1000.0) ** 2 * 40.0
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    keep = np.zeros(count, dtype=bool)
    for index in range(1, count):
        x, y, w, h, area = stats[index]
        if area < min_area:
            continue
        if remove_border_blobs:
            touches_border = x == 0 or y == 0 or x + w >= width or y + h >= height
            oversized = w > 0.5 * width or h > 0.5 * height or area > 0.02 * width * height
            if touches_border and oversized:
                continue
        keep[index] = True
    return keep[labels]


def clean_signature(
    image: np.ndarray,
    sensitivity: float = 1.0,
    min_speck: float = 1.0,
    remove_border_blobs: bool = True,
    ink_colour: str = "black",
) -> CleanResult:
    """Extract the signature strokes from a photo and auto-crop them.

    ``sensitivity`` > 1 keeps fainter strokes; < 1 is stricter. ``min_speck``
    scales the smallest blob (in pixels at 1000 px long side, x40) kept.
    """

    rgb = _downscale(_to_rgb(image))
    gray = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (3, 3), 0)
    darkness = darkness_map(gray)
    threshold = _ink_threshold(darkness, sensitivity)
    strokes = _keep_strokes(darkness > threshold, min_speck, remove_border_blobs)

    height, width = strokes.shape
    if not strokes.any():
        white = np.full((height, width, 3), 255, dtype=np.uint8)
        clear = np.zeros((height, width, 4), dtype=np.uint8)
        return CleanResult(white, clear, False)

    low, high = 0.6 * threshold, 1.6 * threshold
    alpha = np.clip((darkness - low) / (high - low), 0.0, 1.0)
    support = cv2.dilate(strokes.astype(np.uint8), np.ones((3, 3), np.uint8))
    alpha *= support

    ys, xs = np.nonzero(strokes)
    pad = int(round(0.04 * max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)))
    top, bottom = max(0, ys.min() - pad), min(height, ys.max() + pad + 1)
    left, right = max(0, xs.min() - pad), min(width, xs.max() + pad + 1)
    alpha = alpha[top:bottom, left:right]

    colour = np.array(INK_COLOURS.get(ink_colour, INK_COLOURS["black"]), np.float32)
    a = alpha[..., None]
    white_rgb = (255.0 * (1.0 - a) + colour * a).round().astype(np.uint8)
    rgba = np.empty((*alpha.shape, 4), dtype=np.uint8)
    rgba[..., :3] = colour.astype(np.uint8)
    rgba[..., 3] = (alpha * 255.0).round().astype(np.uint8)
    return CleanResult(white_rgb, rgba, True)

In [ ]:
%%writefile /content/ResT/webapp/app.py
"""Phone-friendly Gradio app: signature scanning and ResT pair verification.

    python webapp/app.py --checkpoint checkpoints/rest_gpds_32pairs_10epochs_best.pt
    python webapp/app.py --share          # HTTPS link, needed for phone cameras

The checkpoint can also be given through the REST_CKPT environment variable.
If the model cannot be loaded, the Scan tab still works and Verify is disabled.
"""

from __future__ import annotations

import argparse
import contextlib
import dataclasses
import inspect
import os
from pathlib import Path
import sys
import tempfile
import time
import traceback

import gradio as gr
import numpy as np
from PIL import Image
import torch

WEBAPP_DIR = Path(__file__).resolve().parent
REPO_ROOT = WEBAPP_DIR.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(WEBAPP_DIR))

from cleanup import clean_signature  # noqa: E402
from rest_siamese import ResTConfig, ResTEncoder, SiameseResT  # noqa: E402
from train_cedar import SignatureTransform  # noqa: E402

DEFAULT_CHECKPOINT = REPO_ROOT / "checkpoints" / "rest_gpds_32pairs_10epochs_best.pt"
LFS_POINTER_MAX_BYTES = 10 * 1024
FALLBACK_THRESHOLD = 0.4618
DISCLAIMER = (
    "**Mock-test use only.** This model reached ~75% accuracy (AUC 0.825, EER 25.4%) "
    "on 30 GPDS test writers, and phone photos are out of its training domain. "
    "Do not use the verdict for any real decision."
)


class Verifier:
    """Loads the Siamese ResT checkpoint and scores cleaned signature pairs."""

    def __init__(self, checkpoint: Path, device: str = "auto") -> None:
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        if device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("--device cuda requested but CUDA is not available.")
        self.device = torch.device(device)
        if self.device.type == "cpu":
            torch.set_num_threads(os.cpu_count() or 1)

        payload = self._read_checkpoint(checkpoint)
        known = {field.name for field in dataclasses.fields(ResTConfig)}
        raw_config = payload.get("model_config") or {}
        config_kwargs = {k: v for k, v in raw_config.items() if k in known}
        config_kwargs["gradient_checkpointing"] = False  # saved True; inference-only here
        config = ResTConfig(**config_kwargs)
        state = payload.get("model_state_dict", payload)
        state = {_strip_prefixes(key): value for key, value in state.items()}

        model = SiameseResT(ResTEncoder(config))
        model.load_state_dict(state)
        self.encoder = model.encoder.eval().to(self.device)
        self.transform = SignatureTransform(training=False)

        metrics = payload.get("metrics") or {}
        self.threshold = float(metrics.get("threshold", FALLBACK_THRESHOLD))
        self.eer_threshold = metrics.get("eer_threshold")
        self.epoch = payload.get("epoch")
        self.parameters = sum(p.numel() for p in self.encoder.parameters())

    @staticmethod
    def _read_checkpoint(path: Path) -> dict:
        if not path.is_file():
            raise FileNotFoundError(
                f"Checkpoint not found: {path}. Pass --checkpoint or set REST_CKPT."
            )
        size = path.stat().st_size
        if size < LFS_POINTER_MAX_BYTES:
            raise RuntimeError(
                f"{path} is only {size} bytes, so it is a Git LFS pointer, not the "
                "weights. Run `git lfs pull --include=\"checkpoints/*.pt\"` "
                "(about 388 MB) or point --checkpoint at a downloaded copy."
            )
        try:
            return torch.load(path, map_location="cpu", weights_only=True)
        except Exception:
            # Our own trusted training checkpoint may contain non-tensor objects.
            return torch.load(path, map_location="cpu", weights_only=False)

    def _autocast(self):
        if self.device.type == "cuda":
            return torch.autocast("cuda", dtype=torch.float16)
        return contextlib.nullcontext()

    @torch.inference_mode()
    def distance(self, reference: Image.Image, query: Image.Image) -> tuple[float, float]:
        """Return (L2 distance, milliseconds) for one cleaned pair."""

        batch = torch.stack((self.transform(reference), self.transform(query)))
        batch = batch.to(self.device)
        if self.device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        with self._autocast():
            embeddings = self.encoder(batch).float()
        if self.device.type == "cuda":
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        value = torch.linalg.vector_norm(embeddings[0] - embeddings[1]).item()
        return value, elapsed_ms

    def warm_up(self) -> float:
        blank = Image.new("RGB", (300, 120), "white")
        return self.distance(blank, blank)[1]

    def describe(self) -> str:
        name = str(self.device)
        if self.device.type == "cuda":
            name += f" ({torch.cuda.get_device_name(self.device)}, fp16 autocast)"
        else:
            name += f" ({torch.get_num_threads()} threads, fp32)"
        return name


def _strip_prefixes(key: str) -> str:
    for prefix in ("module.", "_orig_mod."):
        while key.startswith(prefix):
            key = key[len(prefix):]
    return key


def _save_png(array: np.ndarray, stem: str) -> str:
    handle = tempfile.NamedTemporaryFile(prefix=f"{stem}_", suffix=".png", delete=False)
    handle.close()
    Image.fromarray(array).save(handle.name)
    return handle.name


def scan(image, sensitivity, min_speck, remove_border, ink_colour):
    if image is None:
        raise gr.Error("Take or upload a photo of the signature first.")
    result = clean_signature(image, sensitivity, min_speck, remove_border, ink_colour)
    if not result.found_ink:
        raise gr.Error(
            "No ink found. Try raising sensitivity, lowering speck removal, or "
            "retaking the photo closer with even light."
        )
    return (
        result.rgb,
        _save_png(result.rgba, "signature_transparent"),
        _save_png(result.rgb, "signature_white"),
    )


def build_app(verifier: Verifier | None, load_error: str | None) -> gr.Blocks:
    def image_input(label: str) -> gr.Image:
        kwargs = dict(label=label, sources=["upload", "webcam"], type="numpy", height=320)
        # Mirrored webcam captures would flip the signature left-to-right.
        params = inspect.signature(gr.Image.__init__).parameters
        if "webcam_options" in params and hasattr(gr, "WebcamOptions"):
            kwargs["webcam_options"] = gr.WebcamOptions(mirror=False)
        elif "mirror_webcam" in params:
            kwargs["mirror_webcam"] = False
        return gr.Image(**kwargs)

    def cleanup_controls():
        sensitivity = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Sensitivity (higher keeps fainter ink)")
        min_speck = gr.Slider(0.0, 5.0, value=1.0, step=0.1, label="Speck removal")
        remove_border = gr.Checkbox(value=True, label="Remove border blobs (paper edges, shadows)")
        return sensitivity, min_speck, remove_border

    def verify(reference, query, threshold, sensitivity, min_speck, remove_border):
        if verifier is None:
            raise gr.Error(f"Model not loaded: {load_error}")
        if reference is None or query is None:
            raise gr.Error("Provide both the reference and the signature to check.")
        cleaned = []
        for name, image in (("Reference", reference), ("Check", query)):
            result = clean_signature(image, sensitivity, min_speck, remove_border, "black")
            if not result.found_ink:
                raise gr.Error(f"{name} photo: no ink found. Adjust sensitivity or retake it.")
            cleaned.append(result.rgb)
        try:
            value, elapsed_ms = verifier.distance(*(Image.fromarray(c) for c in cleaned))
        except Exception as error:  # surface inference failures in the UI
            traceback.print_exc()
            raise gr.Error(f"Inference failed: {error}") from error

        genuine = value <= threshold
        verdict = "✅ Likely GENUINE (same writer)" if genuine else "❌ Likely FORGED / different writer"
        report = (
            f"## {verdict}\n\n"
            f"| | |\n|---|---|\n"
            f"| Distance | **{value:.4f}** {'≤' if genuine else '>'} threshold {threshold:.4f} |\n"
            f"| Margin | {threshold - value:+.4f} |\n"
            f"| Inference | {elapsed_ms:.0f} ms for the pair |\n"
            f"| Device | {verifier.describe()} |\n\n"
            f"{DISCLAIMER}"
        )
        return report, cleaned[0], cleaned[1]

    with gr.Blocks(title="ResT Signature Scanner") as app:
        gr.Markdown(
            "# ResT signature scanner & verifier\n"
            "On a phone, tap the upload area to open the native camera (sharper than the webcam view). "
            "Fill the frame with the signature and avoid strong shadows."
        )
        with gr.Tab("Scan"):
            scan_image = image_input("Signature photo")
            sensitivity, min_speck, remove_border = cleanup_controls()
            ink_colour = gr.Radio(["black", "blue"], value="black", label="Ink colour")
            scan_button = gr.Button("Clean up", variant="primary")
            scan_preview = gr.Image(label="Cleaned signature", type="numpy", interactive=False)
            with gr.Row():
                transparent_file = gr.File(label="Transparent PNG")
                white_file = gr.File(label="White-background PNG")
            scan_button.click(
                scan,
                [scan_image, sensitivity, min_speck, remove_border, ink_colour],
                [scan_preview, transparent_file, white_file],
                api_name="scan",
            )

        with gr.Tab("Verify"):
            if verifier is None:
                gr.Markdown(f"⚠️ **Verification unavailable — model failed to load.**\n\n```\n{load_error}\n```")
                default_threshold = FALLBACK_THRESHOLD
            else:
                gr.Markdown(
                    f"Model: epoch {verifier.epoch}, {verifier.parameters / 1e6:.1f}M params on "
                    f"{verifier.describe()}. Checkpoint threshold {verifier.threshold:.4f}"
                    + (f" (EER threshold {verifier.eer_threshold:.4f})" if verifier.eer_threshold else "")
                    + "."
                )
                default_threshold = verifier.threshold
            gr.Markdown(DISCLAIMER)
            with gr.Row():
                reference_image = image_input("Reference (known genuine)")
                query_image = image_input("Signature to check")
            threshold = gr.Slider(0.05, 1.5, value=round(default_threshold, 4), step=0.0001, label="Decision threshold (distance ≤ threshold → genuine)")
            with gr.Accordion("Cleanup options", open=False):
                v_sensitivity, v_min_speck, v_remove_border = cleanup_controls()
            verify_button = gr.Button("Verify", variant="primary", interactive=verifier is not None)
            verdict = gr.Markdown()
            with gr.Row():
                reference_clean = gr.Image(label="Cleaned reference", type="numpy", interactive=False)
                query_clean = gr.Image(label="Cleaned check", type="numpy", interactive=False)
            verify_button.click(
                verify,
                [reference_image, query_image, threshold, v_sensitivity, v_min_speck, v_remove_border],
                [verdict, reference_clean, query_clean],
                api_name="verify",
            )
    return app


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--checkpoint", type=Path, default=Path(os.environ.get("REST_CKPT", DEFAULT_CHECKPOINT)))
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    parser.add_argument("--share", action="store_true", help="create a public HTTPS gradio.live link")
    parser.add_argument("--port", type=int, default=7860)
    parser.add_argument("--host", default="0.0.0.0", help="bind address (0.0.0.0 allows LAN access)")
    args = parser.parse_args()

    verifier, load_error = None, None
    try:
        print(f"Loading checkpoint {args.checkpoint} ...", flush=True)
        verifier = Verifier(args.checkpoint, args.device)
        warm_ms = verifier.warm_up()
        print(f"Model ready on {verifier.describe()}; warm-up pair took {warm_ms:.0f} ms; "
              f"threshold={verifier.threshold:.4f}", flush=True)
    except Exception as error:
        load_error = f"{type(error).__name__}: {error}"
        print(f"WARNING: model not loaded, Verify disabled. {load_error}", flush=True)

    build_app(verifier, load_error).queue().launch(
        server_name=args.host, server_port=args.port, share=args.share
    )


if __name__ == "__main__":
    main()

## 3. Get the checkpoint (~388 MB)
Uses `MyDrive/ResT/checkpoints/rest_gpds_32pairs_10epochs_best.pt` from Google Drive if present (fast), otherwise pulls
just that file with Git LFS.

In [ ]:
import os, subprocess
from pathlib import Path

drive_ckpt = Path("/content/drive/MyDrive/ResT/checkpoints/rest_gpds_32pairs_10epochs_best.pt")
repo_ckpt = Path("/content/ResT/checkpoints/rest_gpds_32pairs_10epochs_best.pt")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as error:
    print("Drive not mounted:", error)

if drive_ckpt.is_file() and drive_ckpt.stat().st_size > 10_000:
    ckpt = drive_ckpt
    print("Using checkpoint from Google Drive")
else:
    print("Not on Drive; pulling from Git LFS ...")
    subprocess.run("apt-get -qq install -y git-lfs > /dev/null && git lfs install --skip-smudge", shell=True, check=True)
    subprocess.run(["git", "lfs", "pull", "--include", "checkpoints/rest_gpds_32pairs_10epochs_best.pt"], cwd="/content/ResT", check=True)
    ckpt = repo_ckpt

size = ckpt.stat().st_size
print(f"{ckpt}: {size / 1e6:.1f} MB")
assert size > 10_000, "Still an LFS pointer file: the LFS download failed (bandwidth quota?)."
os.environ["REST_CKPT"] = str(ckpt)

## 4. Install Gradio

In [ ]:
!pip install -q gradio

## 5. Launch
Open the `https://….gradio.live` link on your phone. Stop this cell to shut down.

In [ ]:
!cd /content/ResT && python webapp/app.py --share